# Dataset Insights (Resolution 1024)
This notebook summarizes a generated dataset of elevation-map paths and computes additional insights:
- path crossing heatmap
- elevation distributions
- path length distribution
- normalized relative elevation profile
- normalized Hausdorff distances
- identical path count


## 1. Setup and Configuration
Import libraries, configure solver, and load calibration and dataset summary files.


In [ ]:
import json
import logging
import sys
import multiprocessing
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm
from loguru import logger as loguru_logger

# Define ROOT path first (notebook is in notebooks/)
ROOT = Path("..").resolve()

try:
    from passage.generate import _resolve_solver_config, _assign_splits
except Exception:
    # If import fails (e.g., running in notebook without package installed),
    # temporarily prepend local `src` to sys.path and retry.
    sys.path.insert(0, str(ROOT / "src"))
    from passage.generate import _resolve_solver_config, _assign_splits


# Minimal stubs for linting / quick notebook runs
def process_sample(idx, metadata_dir, tensors_dir, resolution, global_min, global_max):
    """Lightweight placeholder for processing a single sample used for notebook diagnostics."""
    return {"processed": False}


def _process_sample_star(args):
    return process_sample(*args)


# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

# Paths
CONFIG_PATH = ROOT / "config" / "config.yaml"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"
DATASET_ROOT = ROOT / "outputs" / "dataset"
SUMMARY_PATH = DATASET_ROOT / "summary.json"
DEFAULT_COST_MODEL = "elevation"

# Solver config
with open(CONFIG_PATH) as f:
    _config = yaml.safe_load(f) or {}
_solver_cfg = _config.get("solver", {})
SOLVER_CONFIG = _resolve_solver_config(_solver_cfg)

# Calibration
with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)
global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")

NameError: name 'ROOT' is not defined

In [ ]:
# Parameters (papermill)
RESOLUTION = 256
SPLIT = "train"  # train, calibration, validation, test
MAX_SAMPLES = None
MAX_WORKERS = 8
SAVE_WORLD_COVER = True

## 2. Dataset Generation
For each dataset sample, we load the stored paths and per-point elevations from metadata (no recomputation):
- **Straight-line path** between markers (baseline, generated analytically)
- **Stored no-obstacle path** (`paths.without_obstacles`) and **stored obstacle-aware path** (`paths.with_obstacles`)

Metadata now includes per-point elevations and the list of tiles used to build the crop. Metrics are computed for each sample using multiprocessing to speed up processing across CPU cores.

### Metric computations and explanations

- **Path Occurrence Heatmap**: for every sample, we increment a per-pixel counter for each pixel that belongs to the **with_obstacles** stored path. The resulting image shows frequent routing corridors.

- **Covered Tiles Visualization**: we count how often each source tile appears in `tiles_used` and visualize coverage in a lat/lon scatter plot, colored by usage frequency.

- **Start Point Elevation Distribution**: histogram of the elevation at the first pixel of each stored path. This helps understand where paths begin (e.g., high vs low terrain).

- **All Path Points Elevation Distribution**: histogram aggregating elevation values for every path pixel across all samples.

- **Path Length Distribution**: histogram of path lengths measured as number of pixels in the stored path.

- **Normalized Relative Path Elevation Profile**: for each stored path we subtract the start elevation (relative elevation) and resample the sequence to 100 uniformly spaced points in the normalized path domain [0, 1]. We then compute the mean and standard deviation across samples to show a typical elevation trend along a path.

- **Normalized Hausdorff Distance**: we compute the Hausdorff distance $H(P,Q)$ between two paths and normalize by the straight-line (marker) distance $D$ between start and end: normalized = $H(P,Q) / D$. We compute two variants:
  - **Clean vs Straight**: $P$ = path_without_obstacles, $Q$ = straight-line markers path. Measures terrain-induced deviation from straight.
  - **Obstacles vs Clean**: $P$ = path_with_obstacles, $Q$ = path_without_obstacles. Measures deviation specifically introduced by obstacles.

- **Identical Path Count**: number and percentage of samples where the stored obstacle-aware path exactly equals the stored no-obstacle path (exact pixel list equality).

In [ ]:
# Dataset paths and summary (derived from parameters)

RES_DIR = DATASET_ROOT / str(RESOLUTION)
SPLIT_DIR = RES_DIR / SPLIT

METADATA_DIR = SPLIT_DIR / "metadata"

TENSORS_DIR = SPLIT_DIR / "tensors"

IMAGES_DIR = SPLIT_DIR / "images"

SUMMARY_PATH = RES_DIR / "summary.json"

with open(SUMMARY_PATH) as f:
    summary = json.load(f)

# Optional per-tile counts (if present in summary)
tile_counts = summary.get("tile_counts", {}) or {}

print("Dataset Summary:")
print(json.dumps(summary, indent=2))

# Determine number of samples from summary + split assignments
total_requested = int(summary.get("requested_samples", summary.get("num_samples", 0)))
split_cfg = _config.get("splits", {}) or {}
split_enabled = bool(split_cfg.get("enabled", True))
split_ratios = split_cfg.get("ratios", {}) or {"train": 1.0}
split_seed = int(split_cfg.get("seed", 42))
assignments = (
    _assign_splits(total_requested, split_ratios, seed=split_seed)
    if split_enabled
    else {i: "train" for i in range(total_requested)}
)
split_indices = [idx for idx, split in assignments.items() if split == SPLIT]

if MAX_SAMPLES is not None:
    try:
        max_s = int(MAX_SAMPLES)
        split_indices = split_indices[:max_s]
    except Exception:
        pass

num_samples = len(split_indices)
print(f"Using split={SPLIT} with {num_samples} samples (requested={total_requested})")


def resolve_sample_paths(metadata_dir: Path, tensors_dir: Path, idx: int):
    metadata_matches = list(metadata_dir.rglob(f"{idx:010d}.yaml"))
    tensor_matches = list(tensors_dir.rglob(f"{idx:010d}.npy"))
    metadata_path = metadata_matches[0] if metadata_matches else None
    tensor_path = tensor_matches[0] if tensor_matches else None
    return metadata_path, tensor_path


def describe_stats(values, name):
    values = [v for v in values if v is not None]
    if not values:
        print(f"{name}: no data")
        return None
    arr = np.array(values, dtype=np.float64)
    stats = {
        "count": int(arr.size),
        "min": float(np.min(arr)),
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "max": float(np.max(arr)),
        "std": float(np.std(arr)),
    }
    print(f"{name}: {stats}")
    return stats

In [ ]:
# Run processing (parallel or sequential depending on MAX_WORKERS)
# This cell defines `processed` and builds all aggregate statistics.
args = [
    (
        idx,
        METADATA_DIR,
        TENSORS_DIR,
        RESOLUTION,
        global_min,
        global_max,
        # sample-specific fields are read inside process_sample when needed
    )
    for idx in split_indices
]

results = []
if MAX_WORKERS == 1:
    print("Running sequential processing (MAX_WORKERS==1)")
    for a in tqdm(args, total=len(args)):
        results.append(process_sample(*a))
else:
    pool_size = MAX_WORKERS
    print(f"Running with {pool_size} workers")
    if args:
        with multiprocessing.Pool(processes=pool_size) as pool:
            for r in tqdm(pool.imap_unordered(_process_sample_star, args), total=len(args)):
                results.append(r)
    else:
        print("No samples to process (args empty)")

# Filter successful processed entries
processed = [r for r in results if r.get("processed")]

samples_processed = len(processed)

# Defaults for downstream visualizations/stats when no samples are processed
path_crossings_clean = np.zeros((RESOLUTION, RESOLUTION), dtype=np.int64)
path_crossings_obs = np.zeros_like(path_crossings_clean)
start_elevations = []
all_elevations = []
all_path_points = []
path_lengths = []
normalized_profiles = []
hausdorff_clean_straight = []
hausdorff_obs_clean = []
identical_paths_count = 0
tile_loading_seconds = []
marker_search_seconds = []
marker_find_path_without_seconds = []
marker_find_path_with_seconds = []
graph_prebuild_seconds = []
obstacle_generation_seconds = []
cost_compute_seconds = []
tensor_save_seconds = []
metadata_write_seconds = []
image_conversion_seconds = []
image_write_seconds = []
area_ratios = []
cost_diff_percents = []

In [ ]:
# Diagnostics are now appended inside the main aggregation cell above.
# This cell intentionally left as a no-op placeholder.

## 3. Dataset Summary
Report basic statistics: requested vs. processed samples, and success rate.


In [ ]:
summary_stats = {
    "resolution": RESOLUTION,
    "requested_samples": summary.get("requested_samples"),
    "generated_samples": summary.get("generated_samples"),
    "successful_samples": summary.get("successful_samples"),
    "processed_samples": len(processed),
    "processed_rate": (len(processed) / max(num_samples, 1)) * 100,
}

summary_df = pd.DataFrame([summary_stats])
summary_df

## 4. Covered Tiles Visualization
Each point represents a source tile used to build the elevation crops. The color indicates how many samples used the tile.

In [ ]:
# Covered Tiles Visualization: build a 180x360 (1° per pixel) grid image from metadata overlapping tiles
# Zero-based full-globe grid: rows=180 (lat), cols=360 (lon); lat range [-90,89], lon range [-180,179]
if tile_counts:
    grid = np.zeros((180, 360), dtype=np.int32)
    for name, t in tile_counts.items():
        lat = t.get("lat")
        lon = t.get("lon")
        cnt = int(t.get("count", 0))
        if lat is None or lon is None:
            continue
        lat_i = int(round(lat))
        lon_j = int(round(lon))
        # clamp to valid ranges
        lat_i = max(-90, min(89, lat_i))
        lon_j = max(-180, min(179, lon_j))
        i = 89 - lat_i  # row 0 = northmost lat (89)
        j = lon_j + 180
        grid[i, j] = cnt

    if grid.sum() == 0:
        print("No tile coverage to build 180x360 grid.")
    else:
        # Log-normalize non-zero counts for display
        logg = np.log1p(grid.astype(np.float64))
        nz = logg[grid > 0]
        if nz.size == 0:
            print("No nonzero counts after building grid.")
        else:
            vmin = float(nz.min())
            vmax = float(nz.max())

            # Create masked array for display (zeros shown as white)
            masked_logg = np.ma.masked_where(grid == 0, logg)

            # Display with matplotlib
            fig, ax = plt.subplots(figsize=(14, 7))
            cmap = cm.viridis.copy()
            cmap.set_bad(color="white")  # zeros shown as white

            # Use extent to set lat/lon axes
            extent = [-180, 179, -90, 89]  # [left, right, bottom, top]
            im = ax.imshow(
                masked_logg,
                origin="upper",
                extent=extent,
                cmap=cmap,
                aspect="equal",
                vmin=vmin if vmax > vmin else 0,
                vmax=vmax if vmax > vmin else 1,
            )

            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")

            # Add colorbar
            cbar = plt.colorbar(im, ax=ax, shrink=0.6, label="log1p(count)")

            plt.tight_layout()
            plt.show()

            # Save the image and raw counts
            if SAVE_WORLD_COVER:
                out_dir = ROOT / "outputs" / "dataset" / str(RESOLUTION)
                out_dir.mkdir(parents=True, exist_ok=True)
                img_path = out_dir / "world_cover_map.png"
                npy_path = out_dir / "world_cover_tiles.npy"

                # Save figure
                fig_save, ax_save = plt.subplots(figsize=(14, 7))
                im_save = ax_save.imshow(
                    masked_logg,
                    origin="upper",
                    extent=extent,
                    cmap=cmap,
                    aspect="equal",
                    vmin=vmin if vmax > vmin else 0,
                    vmax=vmax if vmax > vmin else 1,
                )
                ax_save.set_xlabel("Longitude")
                ax_save.set_ylabel("Latitude")
                ax_save.set_title("Covered Tiles (180x360 grid, 1° per pixel) — zeros shown as white")
                plt.colorbar(im_save, ax=ax_save, shrink=0.6, label="log1p(count)")
                plt.tight_layout()
                fig_save.savefig(str(img_path), dpi=150)
                plt.close(fig_save)

                np.save(str(npy_path), grid)
                print(f"Saved world cover image to {img_path} and raw counts to {npy_path}")
else:
    print("No tile coverage data available.")

## 4. Path Occurrence Heatmaps
Two heatmaps aggregate pixel crossing counts across the dataset:
- Left: Clean (no obstacles) stored path crossings
- Right: With Obstacles stored path crossings


In [ ]:
# Path occurrence heatmaps (log-scaled) using matplotlib
if samples_processed == 0 or (np.sum(path_crossings_clean) == 0 and np.sum(path_crossings_obs) == 0):
    print("No path crossing data available.")
else:
    clean_log = np.where(path_crossings_clean > 0, np.log1p(path_crossings_clean), np.nan)
    obs_log = np.where(path_crossings_obs > 0, np.log1p(path_crossings_obs), np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    im1 = axes[0].imshow(clean_log, cmap="hot", origin="upper")
    axes[0].set_title("Path Crossing Heatmap - Clean (log scale)")
    axes[0].axis("off")
    plt.colorbar(im1, ax=axes[0], label="log(1+count)")

    im2 = axes[1].imshow(obs_log, cmap="hot", origin="upper")
    axes[1].set_title("Path Crossing Heatmap - With Obstacles (log scale)")
    axes[1].axis("off")
    plt.colorbar(im2, ax=axes[1], label="log(1+count)")

    fig.suptitle(f"Path Crossing Heatmaps ({samples_processed} samples)")
    plt.tight_layout()
    plt.show()

## 5. Start Point Elevation Distribution
Histogram of the absolute elevation at the first point of each obstacle-aware path.

In [ ]:
vals = [v for v in start_elevations if v is not None]
if vals:
    describe_stats(vals, "Start Point Elevations")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("Start Point Elevations")
    plt.tight_layout()
    plt.show()
else:
    print("No start elevations available.")

## 6. All Path Points Elevation Distribution
Histogram of all elevations visited by obstacle-aware paths.

In [ ]:
vals = [v for v in all_elevations if v is not None]
if vals:
    describe_stats(vals, "All Path Point Elevations")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=50, edgecolor="black")
    plt.xlabel("Elevation (m)")
    plt.ylabel("Count")
    plt.title("All Path Point Elevations")
    plt.tight_layout()
    plt.show()
else:
    print("No path elevation data available.")

## 7. Path Length Distribution
Histogram of obstacle-aware path lengths (number of pixels).

In [ ]:
vals = [v for v in path_lengths if v is not None]
if vals:
    describe_stats(vals, "Path Lengths (pixels)")
    plt.figure(figsize=(8, 5))
    plt.hist(vals, bins=30, edgecolor="black")
    plt.xlabel("Length (pixels)")
    plt.ylabel("Count")
    plt.title("Path Lengths")
    plt.tight_layout()
    plt.show()
else:
    print("No path length data available.")

## 8. Normalized Relative Path Elevation Profile
Each path elevation sequence is shifted by its start elevation and resampled to a common length $[0,1]$.
We then plot the mean and standard deviation across samples.

In [ ]:
if normalized_profiles:
    profiles_arr = np.array(normalized_profiles, dtype=np.float64)
    mean_profile = np.mean(profiles_arr, axis=0)
    std_profile = np.std(profiles_arr, axis=0)
    x_axis = np.linspace(0, 1, profiles_arr.shape[1])

    plt.figure(figsize=(10, 6))
    plt.plot(x_axis, mean_profile, "b-", linewidth=2, label="Mean Relative Elevation")
    plt.fill_between(
        x_axis, mean_profile - std_profile, mean_profile + std_profile, alpha=0.2, color="blue", label="Std Dev"
    )
    plt.xlabel("Normalized Path Progress")
    plt.ylabel("Relative Elevation (m)")
    plt.title("Normalized Relative Elevation Profile")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No normalized profiles available.")

## 9. Hausdorff Distance: Path vs. Straight Line
Normalized Hausdorff distance between the optimal (no-obstacle) path and the straight-line path between markers. This measures how much terrain elevation causes deviations from a straight line.

In [ ]:
clean_vals = [v for v in hausdorff_clean_straight if v is not None and v > 0]
if clean_vals:
    describe_stats(clean_vals, "Normalized Hausdorff: Clean vs Straight")
    plt.figure(figsize=(8, 5))
    plt.hist(clean_vals, bins=30, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Clean vs Straight Line (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for clean vs straight.")

## 10. Hausdorff Distance: Obstacles vs. No Obstacles
Normalized Hausdorff distance between the obstacle-aware path and the no-obstacle path on the same elevation map.

In [ ]:
obs_vals = [v for v in hausdorff_obs_clean if v is not None and v > 0]
if obs_vals:
    describe_stats(obs_vals, "Normalized Hausdorff: Obstacles vs Clean")
    plt.figure(figsize=(8, 5))
    plt.hist(obs_vals, bins=100, edgecolor="black")
    plt.xlabel("Normalized Distance")
    plt.ylabel("Count")
    plt.title("Normalized Hausdorff: Obstacles vs No Obstacles (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive Hausdorff data for obstacles vs clean.")

## 11. Area Ratio & Cost Difference
Area ratio = (area between paths in pixels) / (marker distance in pixels)^2. Cost difference = 100 * (cost_with - cost_without) / cost_without.

In [ ]:
# Area ratio distribution and cost difference visualization
area_pos = [v for v in area_ratios if v is not None and v > 0]
if area_pos:
    describe_stats(area_pos, "Area Ratio (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(area_pos, bins=40, edgecolor="black")
    plt.xlabel("Area Ratio")
    plt.ylabel("Count")
    plt.title("Area Ratio: (area between paths) / (marker_dist^2) (values > 0)")
    plt.tight_layout()
    plt.show()
else:
    print("No positive area ratio data available.")

# Cost difference percentage distribution
cd_vals = [v for v in cost_diff_percents if v is not None and v > 0]
if cd_vals:
    describe_stats(cd_vals, "Cost Difference Percent (positive)")
    plt.figure(figsize=(8, 5))
    plt.hist(cd_vals, bins=50, edgecolor="black")
    plt.xlabel("Percent")
    plt.ylabel("Count")
    plt.title("Cost Difference Percent: 100*(cost_with - cost_without)/cost_without")
    plt.tight_layout()
    plt.show()
else:
    print("No cost difference percentage data available.")

## 11b. Timing Summary

Plot per-sample timing distributions to identify hotspots:
- `tile_loading_seconds`
- `marker_search_seconds`
- `marker_find_path_without_seconds`
- `marker_find_path_with_seconds`

- `graph_prebuild_seconds`

- `obstacle_generation_seconds`- `image_write_seconds`

- `cost_compute_seconds`- `image_conversion_seconds`

- `tensor_save_seconds`- `metadata_write_seconds`

In [ ]:
# Timing distributions and stats
gp_vals = [v for v in graph_prebuild_seconds if v is not None and v >= 0]
og_vals = [v for v in obstacle_generation_seconds if v is not None and v >= 0]
cc_vals = [v for v in cost_compute_seconds if v is not None and v >= 0]
tl_vals = [v for v in tile_loading_seconds if v is not None and v >= 0]
ms_vals = [v for v in marker_search_seconds if v is not None and v >= 0]
mpw_vals = [v for v in marker_find_path_without_seconds if v is not None and v >= 0]
mpt_vals = [v for v in marker_find_path_with_seconds if v is not None and v >= 0]
ts_vals = [v for v in tensor_save_seconds if v is not None and v >= 0]
mw_vals = [v for v in metadata_write_seconds if v is not None and v >= 0]
ic_vals = [v for v in image_conversion_seconds if v is not None and v >= 0]
iw_vals = [v for v in image_write_seconds if v is not None and v >= 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if gp_vals:
    axes[0].hist(gp_vals, bins=30, edgecolor="black")
    axes[0].set_title("Graph prebuild (s)")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Count")
else:
    axes[0].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[0].set_title("Graph prebuild (s)")

if og_vals:
    axes[1].hist(og_vals, bins=30, edgecolor="black")
    axes[1].set_title("Obstacle generation (s)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Count")
else:
    axes[1].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[1].set_title("Obstacle generation (s)")

if cc_vals:
    axes[2].hist(cc_vals, bins=30, edgecolor="black")
    axes[2].set_title("Cost compute (s)")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Count")
else:
    axes[2].text(0.5, 0.5, "No data", ha="center", va="center")
    axes[2].set_title("Cost compute (s)")

fig.suptitle("Per-sample Timing Distributions")
plt.tight_layout()
plt.show()

# Additional timing plots for new metrics (if present)
new_metrics = [
    ("Tile load", tl_vals),
    ("Marker search", ms_vals),
    ("Marker find (no obstacles)", mpw_vals),
    ("Marker find (with obstacles)", mpt_vals),
    ("Tensor save", ts_vals),
    ("Metadata write", mw_vals),
    ("Image conversion", ic_vals),
    ("Image write", iw_vals),
]
new_metrics = [(name, vals) for name, vals in new_metrics if vals]
if new_metrics:
    n2 = len(new_metrics)
    cols2 = 3
    rows2 = math.ceil(n2 / cols2)
    fig2, axes2 = plt.subplots(rows2, cols2, figsize=(5 * cols2, 4 * rows2))
    ax_list2 = axes2.flatten() if hasattr(axes2, "flatten") else [axes2]
    for ax in ax_list2[n2:]:
        ax.axis("off")
    for ax, (name, vals) in zip(ax_list2, new_metrics):
        ax.hist(vals, bins=30, edgecolor="black")
        ax.set_title(f"{name} (n={len(vals)})")
        ax.set_xlabel("seconds")
    fig2.suptitle("Additional Timing Distributions")
    plt.tight_layout()
    plt.show()

    # Print summary stats for the timing buckets (if any exist)
    if not (gp_vals or og_vals or cc_vals):
        print("No timing data available.")
    else:
        if gp_vals:
            describe_stats(gp_vals, "Graph prebuild seconds")
        if og_vals:
            describe_stats(og_vals, "Obstacle generation seconds")
        if cc_vals:
            describe_stats(cc_vals, "Cost compute seconds")

## 12. Identical Path Count
Number of cases where the obstacle-aware path is exactly the same as the no-obstacle path.

In [ ]:
if samples_processed > 0:
    pct_identical = (identical_paths_count / samples_processed) * 100
else:
    pct_identical = 0.0

print(f"Identical paths: {identical_paths_count} / {samples_processed} ({pct_identical:.1f}%)")